# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Outcome B (code-only): TexCot22 (doi:10.18738/T8/5M9NCI) is inaccessible through every authorized public route. dataverse.tdl.org CloudFront/WAF returns HTTP 403 'Request blocked' to programmatic clients from both DGX and a fresh Colab CPU VM (urllib and requests, identifying UA; Request ID o_7amV28nPiajjqyOUqMmMYIG-R4K8bURV7oN8paL6BI3QdLvqQYEA==). No public mirror exists (HF/Zenodo/GitHub/Harvard-DataCite checked). Author code pinned (7c25278b33d090a0ec68f6279fe25034ae4ad2fc) and dependency graph verified on Colab CPU; deliverable is a statically validated code-only notebook with an execution guard, never executed.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: NTrack: A Multiple-Object Tracker and Dataset for Infield Cotton Boll Counting

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-ab3aad79edeaf72b684b184a1da8a013)


# NTrack — Cotton Boll Tracking and Counting on TexCot22 (CODE-ONLY / NOT EXECUTED)

Reproduction of **M. A. Al Muzaddid and W. J. Beksi, "NTrack: A Multiple-Object Tracker and
Dataset for Infield Cotton Boll Counting," arXiv:2312.10922v1 (IEEE T-ASE 21(9):7452-7464, 2024,
DOI 10.1109/TASE.2023.3342791)**.

- Author code (Apache-2.0): https://github.com/robotic-vision-lab/NTrack-A-Multiple-Object-Tracker
  pinned commit `7c25278b33d090a0ec68f6279fe25034ae4ad2fc`
- Dataset TexCot22 (CC BY-NC-SA 4.0): https://doi.org/10.18738/T8/5M9NCI (Texas Data Repository)
- Project site: https://robotic-vision-lab.github.io/ntrack

**Planned scope (one example):** run the authors' NTrack tracker on the TexCot22 test sequence
`vid25_01` using the dataset's own precomputed detections (`det.txt`), count unique cotton bolls
(unique track IDs), and compare with the paper's Table IV (`vid25_01`: GT 68, NTrack 67),
plus the authors' MOT-metric summary (200 px frame margin, Sec. V-B). Detection training is out
of scope; no detector weights are needed because `det.txt` is provided.

## ⚠️ CODE-ONLY — this notebook has never been executed / 本体は実行されていません

**English.** This notebook is delivered as **CODE-ONLY / NOT EXECUTED**. The essential input — the
TexCot22 dataset — could not be obtained through any authorized public route during authoring:
`dataverse.tdl.org` is fronted by AWS CloudFront + WAF, which returned **HTTP 403 "Request blocked"**
to every programmatic client tested (curl on the authoring host; `urllib` and `requests` from a
fresh Google Colab VM, with an identifying User-Agent; Request ID
`o_7amV28nPiajjqyOUqMmMYIG-R4K8bURV7oN8paL6BI3QdLvqQYEA==`). No public mirror of TexCot22 exists
(checked Hugging Face, Zenodo, Kaggle, GitHub, Harvard Dataverse harvested metadata, DataCite).
The first code cell raises `RuntimeError` so that *Run all* cannot produce a misleading result.
It becomes runnable only when the dataset is supplied (see the acquisition cells for the
manual-browser route). **No scientific claim below has been validated by execution.**

**日本語.** 本ノートブックは **CODE-ONLY（未実行）** です。必須入力である TexCot22 データセットを、
認められた公開経由では取得できませんでした（`dataverse.tdl.org` の CloudFront/WAF が、作成ホストと
新規 Colab VM の双方のプログラム的クライアントに HTTP 403 を返却）。公開ミラーも確認できませんでした。
最初のコードセルが `RuntimeError` を発生させるため、*Run all* は誤った結果を出力できません。
データセットを手動取得した場合にのみ実行可能です。以下の科学的主張は実行検証されていません。

**Provenance.** The author's Python code is reused **verbatim** at the pinned commit; the only
modifications are the documented compatibility edits in the patch cell. Notebook glue cells are
AI-authored and labeled. The authors did not provide this Colab notebook.

---

## Runtime / ランタイム

**CPU only**（メニュー: 「ランタイム」→「ランタイムのタイプの変更」→ CPU）。
The tracking path uses OpenCV CPU operators only (Farneback optical flow, Lap-style assignment,
1000-particle particle filters); the pinned code contains no CUDA/GPU path, so a GPU adds nothing.
Expected (estimate, not measured): pip setup ~2-3 min; data download depends on route; tracking
one ~300-frame 4K sequence ~5-10 min on a standard Colab CPU VM.

### Execution guard / 実行ガード

This cell makes accidental *Run all* impossible while the essential dataset is missing. Delete or
bypass this cell **only** after TexCot22 has been supplied by an authorized route (e.g., your own
manual browser download, or a fixed WAF configuration).

In [ ]:
GUARD_MESSAGE = (
    "CODE-ONLY / NOT EXECUTED notebook: the essential TexCot22 dataset "
    "(doi:10.18738/T8/5M9NCI) was inaccessible through every authorized public route "
    "at authoring time (dataverse.tdl.org CloudFront/WAF returns HTTP 403 to programmatic "
    "clients). Supply the dataset via the documented manual route, then remove this guard."
)
raise RuntimeError(GUARD_MESSAGE)

### Dependencies / 依存関係

Installs the exact import graph of the tracking path (verified on a Colab CPU VM on 2026-10-07:
numpy 2.1, scipy 1.16, opencv 5.0, lap, cython_bbox, motmetrics 1.4, filterpy all importable).
`torch`/`tensorflow` are **not** needed: the detector is bypassed via `det.txt`.
必要な依存のみをインストールします（検出器は det.txt を利用するため torch/tensorflow 不要）。

In [ ]:
%pip install -q lap cython-bbox motmetrics filterpy remotezip PyYAML

### Acquire the author's code at the pinned commit / 著者コードの取得（ピン留めコミット）

Partial clone with blob:none + sparse checkout of `src/`, explicitly fetching the pinned commit
in detached HEAD and asserting it (repository provenance is not trusted from a mutable branch).
著者リポジトリを部分クローンし、指定コミットを detached HEAD で取得・検証します。

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/robotic-vision-lab/NTrack-A-Multiple-Object-Tracker.git"
PINNED_COMMIT = "7c25278b33d090a0ec68f6279fe25034ae4ad2fc"  # README documents the run command
REPO_DIR = "/content/NTrack"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--no-checkout", "--filter=blob:none", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "sparse-checkout", "init", "--cone"], check=True)
subprocess.run(["git", "sparse-checkout", "set", "src"], check=True)
subprocess.run(["git", "fetch", "--filter=blob:none", "origin", PINNED_COMMIT], check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
assert head == PINNED_COMMIT, head
os.chdir(os.path.join(REPO_DIR, "src"))
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("checked out", head)

### Compatibility patch / 互換性パッチ（documented adaptations)

Two material observations from the pinned code, both recorded in the authoring report:

1. `np.float` was removed in NumPy >= 1.24 and appears 6x on the tracking path
   (`src/matching.py` 5x, `src/byte_tracker.py` 1x). We replace it with the built-in `float`
   (identical semantics; `\b` regex avoids `np.float32/64`).
2. `src/utils/opticalflow_util.py::Opticalflow_Manager.get_dense_flow` computes the Farneback
   flow but **does not return it** (`return` statement missing). With the pinned code as-is the
   particle-filter optical-flow predict branch is silently skipped (the particle filter is still
   updated by matched detections and by the neighbor-based relocation of dormant tracks, i.e.
   the paper's RLA). We run the author's code **exactly as pinned** for the primary result
   because Table IV was produced by the authors' code; this observation is documented, not fixed.

互換性修正: (1) NumPy>=1.24 で削除された `np.float` を `float` へ置換（意味は同一）。
(2) `get_dense_flow` が flow を return していない事実を確認済み。Table IV が著者コードの
挙動で作成されたことに鑑み、主結果はピン留めコードのまま実行し、観察のみを記録します。

In [ ]:
import re
from pathlib import Path

for rel in ["src/matching.py", "src/byte_tracker.py"]:
    p = Path(REPO_DIR) / rel
    t = p.read_text()
    patched, n = re.subn(r"np\.float\b", "float", t)
    assert n >= 1, (rel, "np.float not found")
    p.write_text(patched)
    print(f"patched {rel}: {n} occurrence(s) of np.float -> float")

### TexCot22 acquisition — route A (programmatic Dataverse) / データ取得 経路A

Texas Data Repository (Dataverse) record `doi:10.18738/T8/5M9NCI` — version 2.4, ten ZIP files
(0.95-4.2 GB each; verified via DataCite metadata `10.18738/T8/5M9NCI`). Route A queries the
Dataverse API for the file listing, then reads each tracking ZIP's **central directory only**
(HTTP range requests via `remotezip`) to find which archive contains `test/vid25_01`, and
extracts just that sequence. **At authoring time this route returned HTTP 403 from both the
authoring host and a fresh Colab VM** (CloudFront "Request blocked"). It is kept because a future
WAF change would make it work unchanged. 経路A: Dataverse API + HTTP レンジ要求で ZIP の中央
ディレクトリのみを読み、必要なシーケンスだけ展開します（作成時点では 403）.

In [ ]:
import json, urllib.parse
import requests

DATAVERSE = "https://dataverse.tdl.org"
DOI = "doi:10.18738/T8/5M9NCI"
UA = {"User-Agent": "PhenoPaperColabReproduction/1.0 (research notebook authoring)"}

resp = requests.get(f"{DATAVERSE}/api/datasets/:persistentId",
                    params={"persistentId": DOI}, headers=UA, timeout=120)
if resp.status_code == 403:
    raise RuntimeError(
        "dataverse.tdl.org returned 403 to this programmatic client (CloudFront/WAF). "
        "Use the manual browser route in the next cell."
    )
resp.raise_for_status()
files = resp.json()["data"]["latestVersion"]["files"]
listing = {f["dataFile"]["filename"]: (f["dataFile"]["id"], f["dataFile"]["filesize"],
                                       f["dataFile"].get("md5")) for f in files}
for name, (fid, size, md5) in sorted(listing.items()):
    print(f"{fid:>9}  {size:>13,}  {md5}  {name}")
json.dump(listing, open("/content/texcot22_file_listing.json", "w"))

### Route A continued: locate and extract only `test/vid25_01` / 必要部分のみ展開

Reads ZIP central directories via HTTP ranges (kilobytes, not gigabytes), extracts only the
sequence directory `test/vid25_01` (images + `gt/gt.txt` + `det/det.txt`). If range requests are
unsupported by the server, fall back to a full single-ZIP download with MD5 verification and
extract the sequence from it (disclosed as a large download; the ZIP is deleted after
extraction).

In [ ]:
import hashlib, zipfile
from remotezip import RemoteZip

listing = json.load(open("/content/texcot22_file_listing.json"))
SEQ = "vid25_01"
tracking_zips = {n: v for n, v in listing.items()
                 if n.lower().startswith("texcot22-")}
assert tracking_zips, "no TexCot22-*.zip in listing"

def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

os.makedirs("/content/TexCot22", exist_ok=True)
extracted = False
for name, (fid, size, md5) in sorted(tracking_zips.items(), key=lambda kv: kv[1][1]):
    dfurl = f"{DATAVERSE}/api/access/datafile/{fid}"
    try:
        with RemoteZip(dfurl, headers=UA) as z:
            members = [m for m in z.namelist() if f"/{SEQ}/" in f"/{m}"]
            if not members:
                print(name, "does not contain", SEQ)
                continue
            print(name, "->", len(members), "members for", SEQ)
            z.extractall("/content/TexCot22", members=members)
        extracted = True
        break
    except Exception as e:
        print(name, "range route failed:", type(e).__name__, str(e)[:200])

if not extracted:
    # Fallback: full download of one ZIP (large!), MD5-checked, sequence extracted, ZIP deleted.
    for name, (fid, size, md5) in sorted(tracking_zips.items(), key=lambda kv: kv[1][1]):
        dfurl = f"{DATAVERSE}/api/access/datafile/{fid}"
        r = requests.get(dfurl, headers=UA, stream=True, timeout=3600)
        r.raise_for_status()
        zpath = f"/content/{name}"
        with open(zpath, "wb") as f:
            for blk in r.iter_content(1 << 20):
                f.write(blk)
        got = md5_of(zpath)
        assert md5 and got == md5, (name, got, md5)
        with zipfile.ZipFile(zpath) as z:
            members = [m for m in z.namelist() if f"/{SEQ}/" in f"/{m}"]
            assert members, (name, "sequence not inside")
            z.extractall("/content/TexCot22", members=members)
        os.remove(zpath)
        extracted = True
        print("extracted", SEQ, "from", name, f"({size/1e9:.2f} GB, md5 ok)")
        break
assert extracted, "could not obtain vid25_01 by any route"

### Route B (manual, documented fallback) / 手動取得フォールバック

If both automated routes are blocked by the WAF (the situation at authoring time), a human may
download the ZIP in a normal browser — interactive access is not something this notebook may
automate — and then either upload it or mount Drive. Keep the archive under `/content`; do not
place dataset bytes on the authoring host. 手動でのブラウザ取得のみが可能な場合は、ZIP を
`/content` に配置してください（データセット本体を作業ホストに置かないこと）。

In [ ]:
# Manual route — fill in after a browser download, e.g. via the Colab file upload dialog:
# from google.colab import files
# uploaded = files.upload()                # choose TexCot22-<N>.zip (may be ~4 GB)
# Then verify MD5 against the Dataverse listing and extract only test/vid25_01 as above.
# A Drive mount is an alternative: files.upload() is slow for multi-GB archives.
raise RuntimeError("Route B is a documented manual fallback; see the Markdown cell above.")

### Verify the downloaded bytes / ダウンロード検証

Validates the actual sequence directory before scientific use: expected subdirectories, frame
count, image decodability, and the MOT-format `gt.txt` / `det.txt` column counts
(`<frame>,<id>,<x>,<y>,<w>,<h>,...`).

In [ ]:
seq_dir = "/content/TexCot22/test/vid25_01"
img_dir = os.path.join(seq_dir, "img1")
frames = sorted(os.listdir(img_dir))
assert len(frames) > 0, "no frames"
import cv2
img0 = cv2.imread(os.path.join(img_dir, frames[0]))
assert img0 is not None, "first frame failed to decode"
gt_lines = open(os.path.join(seq_dir, "gt", "gt.txt")).read().strip().splitlines()
det_lines = open(os.path.join(seq_dir, "det", "det.txt")).read().strip().splitlines()
gt_ids = {int(l.split(",")[1]) for l in gt_lines}
print(f"frames: {len(frames)} | first frame shape: {img0.shape} (H, W, C; author code assumes 3840x2160)")
print(f"gt.txt rows: {len(gt_lines)} | unique GT ids (C_gt, Table IV ref: 68): {len(gt_ids)}")
print(f"det.txt rows: {len(det_lines)} | columns: {len(det_lines[0].split(','))}")

### Input preview (before any tracking) / 入力プレビュー

Shows one actual `vid25_01` frame with the dataset's ground-truth boxes (green, reference
annotation) and the precomputed Cascade R-CNN detections from `det.txt` (red, author-provided
inputs, confidence >= 0.6 `track_thresh`). Drawing glue is AI-authored; it follows the drawing
conventions of the author's `src/utils/draw_util.py` (whose `show_output` calls `cv.imshow` and
therefore cannot run headless).

In [ ]:
import matplotlib.pyplot as plt

def draw_boxes(img_bgr, rows, color, max_rows=2000):
    out = img_bgr.copy()
    for r in rows[:max_rows]:
        x, y, w, h = int(r[2]), int(r[3]), int(r[4]), int(r[5])
        cv2.rectangle(out, (x, y), (x + w, y + h), color, 3)
    return out

def frame_rows(path, frame_no=1, min_conf=0.0):
    rows = [l.split(",") for l in open(path).read().strip().splitlines()]
    return [r for r in rows if int(r[0]) == frame_no and float(r[6]) >= min_conf]

img0 = cv2.imread(os.path.join(img_dir, frames[0]))
overlay = draw_boxes(img0, frame_rows(os.path.join(seq_dir, "gt", "gt.txt"), 1), (0, 255, 0))
overlay = draw_boxes(overlay, frame_rows(os.path.join(seq_dir, "det", "det.txt"), 1, 0.6), (0, 0, 255))
plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("vid25_01 frame 1 - green: reference GT boxes | red: det.txt (conf>=0.6)")
os.makedirs("/content/out", exist_ok=True)
plt.savefig("/content/out/vid25_01_input_gt_det.png", bbox_inches="tight", dpi=120)
plt.show()

### Core method: NTrack tracking of `vid25_01` / コア実行

Runs the authors' `do_track` (src/ntrack.py, pinned) on the single sequence with their documented
defaults (`--use_pf True`, `track_thresh 0.6`, `match_thresh 0.9`, `track_buffer 60`), selected
via the authors' own `CottonSequence` loader. `save_vid=False` only skips the optional per-frame
AVI export; it does not change tracking. The particle filter (1000 particles), relative-location
relocation (RLA) and BYTETracker-style association all come from the pinned author files.
Expected: per-sequence MOT-metric summary printed by the authors' `metric_calculator`
(200 px margin on 3840-wide frames, per paper Sec. V-B).

In [ ]:
import argparse
from ntrack import do_track  # pinned author entry point (src/ntrack.py)
from utils import cotton_dataset

args = argparse.Namespace(track_thresh=0.6, match_thresh=0.9, track_buffer=60,
                          min_box_area=100, mot20=False)
seq = cotton_dataset.CottonSequence("vid25_01", "/content/TexCot22/test")
seq_name, metric_acc, results = do_track(args, seq, save_vid=False, use_pf=True)
print("sequence:", seq_name, "| result rows:", len(results))

### Cotton boll counting vs Table IV / 個体数カウント比較

Counts unique IDs in the tracking output (`C_h`, paper Sec. V-C eqs. 11-12: "the number of
unique cotton bolls ... by NTrack") and in `gt.txt` (`C_gt`). Reference from the paper's
Table IV for `vid25_01`: **GT 68, NTrack 67 (1%)**. A single-sequence match is a plausibility
check, not a reproduction of the paper's aggregate MAPE 4.00% / RMSE 4.73 over all 13 test
sequences.

In [ ]:
import pandas as pd

def count_unique_mot_rows(rows):
    return len({int(r[1]) for r in rows})

c_h = count_unique_mot_rows(results)          # unique track IDs in NTrack output
gt_rows = [l.split(",") for l in open(os.path.join(seq_dir, "gt", "gt.txt")).read().strip().splitlines()]
c_gt = len({int(r[1]) for r in gt_rows})      # unique GT object IDs
ref = {"vid25_01": (68, 67)}                  # Table IV
print(pd.DataFrame(
    [[seq_name, c_gt, c_h, ref[seq_name][0], ref[seq_name][1],
      abs(c_h - ref[seq_name][1])]],
    columns=["sequence", "C_gt (computed)", "C_h (computed)",
             "GT (Table IV)", "NTrack (Table IV)", "abs diff vs Table IV"]))
pd.DataFrame([[seq_name, c_gt, c_h]],
             columns=["sequence", "gt_unique_ids", "ntrack_unique_ids"]
             ).to_csv("/content/out/vid25_01_count.csv", index=False)

### MOT metrics summary / 追跡指標

The authors' `do_track` already printed the per-sequence MOT-metric summary through
`utils.metric_calculator.populate_result` (motmetrics `motchallenge_metrics`; boxes within the
200 px frame margin excluded, `max_iou=0.65` matching). This cell re-renders the accumulator for
the record. HOTA is not included because the repository does not bundle a HOTA implementation
(recorded limitation of the paper's code).

In [ ]:
import motmetrics as mm
from utils import metric_calculator

metric_calculator.populate_result(seq_name, metric_acc)

### Tracking output overlay / 追跡結果の可視化

Displays one tracked frame with the tracker's predicted boxes and stable track IDs
(AI-authored glue mirroring the author's `draw_util.draw_rect_with_label` overlay style; the
author's `show_output` cannot run headless because of `cv.imshow`). Label meaning: numbered
colored boxes = NTrack predictions (not reference annotations).

In [ ]:
from ntrack import get_track_bb  # pinned author helper (PF-center boxes with IDs)

frame_no = 1
frame_data = seq[frame_no - 1]
frame_rgb = frame_data["img"]  # skimage RGB
# Re-track a single frame is not possible; instead re-use the stored result rows for frame 1.
rows = [r for r in results if int(r[0]) == frame_no]
vis = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
for r in rows:
    x, y, w, h = int(r[2]), int(r[3]), int(r[4]), int(r[5])
    cv2.rectangle(vis, (x, y), (x + w, y + h), (225, 255, 0), 3)
    cv2.putText(vis, str(int(r[1])), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (225, 255, 0), 3)
plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("vid25_01 frame 1 - NTrack predictions with track IDs (author output rows)")
plt.savefig("/content/out/vid25_01_ntrack_overlay.png", bbox_inches="tight", dpi=120)
plt.show()

## Interpretation, limitations, validation record / 解釈と限界

**English.** This notebook was authored but **never executed**; all figures, counts and metric
values that it would print are unknown at authoring time and nothing on this page is evidence.
When executed with the dataset available, the scientific meaning of the counting cell is limited
to one sequence: Table IV aggregates 13 test sequences (paper MAPE 4.00%, RMSE 4.73); a
single-sequence match or mismatch is a plausibility check only. Known code-level observations
recorded for transparency: the `np.float` patch; the missing `return` in
`get_dense_flow` (optical-flow-guided PF prediction silently skipped in the pinned code; the PF
still updates from detections and RLA relocation); the author's loader reads skimage RGB frames
but passes them to OpenCV as if BGR (kept as-is). Detection training, the 13-sequence aggregate,
baselines (DeepSORT/Tracktor/ByteTrack/TrackFormer) and HOTA are out of scope.

**日本語.** 本ノートブックは作成のみで未実行のため、表示される数値・図は未検証です。カウント比較は
単一シーケンスのものであり、Table IV（13 シーケンス、MAPE 4.00%/RMSE 4.73）の再現ではありません。
既知の観察: `np.float` パッチ、`get_dense_flow` の return 欠落（光フローによる PF 予測は
スキップされる）、RGB/BGR の扱い。検出器学習・ベースライン・HOTA は対象外です。

### Validation record / 検証記録

| Item | Status |
| --- | --- |
| Author code pinned and inspected | yes — commit `7c25278b33d090a0ec68f6279fe25034ae4ad2fc` |
| Dependency import graph verified on Colab CPU VM | yes — 2026-10-07 diagnostic probe (lap, cython_bbox, motmetrics, filterpy import OK) |
| TexCot22 bytes validated | **NO — inaccessible (CloudFront/WAF 403 on all programmatic routes; no public mirror found)** |
| Notebook executed | **NO — outcome B, code-only** |

### References

1. M. A. Al Muzaddid, W. J. Beksi, "NTrack: A Multiple-Object Tracker and Dataset for Infield
   Cotton Boll Counting," arXiv:2312.10922v1; IEEE T-ASE 21(9):7452-7464, 2024.
2. Author repository, pinned commit `7c25278b33d090a0ec68f6279fe25034ae4ad2fc` (Apache-2.0).
3. TexCot22, Texas Data Repository, `doi:10.18738/T8/5M9NCI` (CC BY-NC-SA 4.0) — *not obtained*.
4. Table IV reference values transcribed from the arXiv v1 HTML (Sec. V-C).